In [1]:
'''Imports'''

import pennylane as qml
from pennylane.qchem import one_particle, two_particle, observable
from pennylane import numpy as pnp

from pyscf import gto, scf, fci, ao2mo, cc
from pyscf.fci import direct_spin1, addons
import numpy as np


from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

In [2]:
class FCISpectralMoments:
    """
    Exact full-space FCI spectral moments in a chosen MO basis.

    Computes

        T_h[n,i,j]
        =
        sum_sigma
        <Psi0|
        a^dag_{i sigma}
        (E0 - H)^n
        a_{j sigma}
        |Psi0>

    and

        T_p[n,i,j]
        =
        sum_sigma
        <Psi0|
        a_{i sigma}
        (H - E0)^n
        a^dag_{j sigma}
        |Psi0>

    directly in PySCF CI space.
    """

    def __init__(
        self,
        mf,
        mo_coeff=None,
    ):
        self.mf = mf
        self.mol = mf.mol

        if mo_coeff is None:
            mo_coeff = mf.mo_coeff

        self.mo_coeff = mo_coeff

        self.nmo = mo_coeff.shape[1]
        self.nelec = self.mol.nelec

        self.nalpha = self.nelec[0]
        self.nbeta = self.nelec[1]

        self.ecore = self.mol.energy_nuc()

        # --------------------------------------------------
        # Build MO integrals
        # --------------------------------------------------

        self.h1 = (
            mo_coeff.T
            @ mf.get_hcore()
            @ mo_coeff
        )

        eri = ao2mo.kernel(
            self.mol,
            mo_coeff,
        )

        self.eri = ao2mo.restore(
            1,
            eri,
            self.nmo,
        )

        # --------------------------------------------------
        # Exact FCI ground state
        # --------------------------------------------------

        self.solver = direct_spin1.FCI(
            self.mol
        )

        self.e0_total = None
        self.e0_electronic = None
        self.ci0 = None

    # ======================================================
    # Exact FCI
    # ======================================================

    def run_fci(self):

        e0, ci0 = self.solver.kernel(
            self.h1,
            self.eri,
            self.nmo,
            self.nelec,
            ecore=self.ecore,
        )

        self.e0_total = float(e0)
        self.e0_electronic = (
            self.e0_total
            - self.ecore
        )

        self.ci0 = ci0

        return {
            "fci_energy": self.e0_total,
            "fci_electronic_energy":
                self.e0_electronic,
            "fci_ci": self.ci0,
        }

    # ======================================================
    # Hamiltonian action
    # ======================================================

    def _apply_h(
        self,
        ci_vec,
        nelec_sector,
    ):
        """
        Apply the electronic Hamiltonian to a CI vector
        in an arbitrary particle-number sector.
        """

        h2eff = direct_spin1.absorb_h1e(
            self.h1,
            self.eri,
            self.nmo,
            nelec_sector,
            fac=0.5,
        )

        return direct_spin1.contract_2e(
            h2eff,
            ci_vec,
            self.nmo,
            nelec_sector,
        )

    # ======================================================
    # Spectral moments
    # ======================================================

    def compute_moments(
        self,
        nmom=10,
    ):

        if self.ci0 is None:
            self.run_fci()

        ci0 = self.ci0
        E0 = self.e0_electronic

        shape = (
            nmom + 1,
            self.nmo,
            self.nmo,
        )

        hole_alpha = np.zeros(
            shape,
            dtype=complex,
        )

        hole_beta = np.zeros(
            shape,
            dtype=complex,
        )

        particle_alpha = np.zeros(
            shape,
            dtype=complex,
        )

        particle_beta = np.zeros(
            shape,
            dtype=complex,
        )

        # ==================================================
        # Alpha hole
        # ==================================================

        if self.nalpha > 0:

            sector = (
                self.nalpha - 1,
                self.nbeta,
            )

            seeds = [
                addons.des_a(
                    ci0,
                    self.nmo,
                    self.nelec,
                    j,
                )
                for j in range(self.nmo)
            ]

            vecs = [
                x.copy()
                for x in seeds
            ]

            for n in range(nmom + 1):

                for i in range(self.nmo):
                    for j in range(self.nmo):

                        hole_alpha[
                            n, i, j
                        ] = np.vdot(
                            seeds[i],
                            vecs[j],
                        )

                if n == nmom:
                    break

                for j in range(self.nmo):

                    Hvec = self._apply_h(
                        vecs[j],
                        sector,
                    )

                    vecs[j] = (
                        E0 * vecs[j]
                        - Hvec
                    )

        # ==================================================
        # Beta hole
        # ==================================================

        if self.nbeta > 0:

            sector = (
                self.nalpha,
                self.nbeta - 1,
            )

            seeds = [
                addons.des_b(
                    ci0,
                    self.nmo,
                    self.nelec,
                    j,
                )
                for j in range(self.nmo)
            ]

            vecs = [
                x.copy()
                for x in seeds
            ]

            for n in range(nmom + 1):

                for i in range(self.nmo):
                    for j in range(self.nmo):

                        hole_beta[
                            n, i, j
                        ] = np.vdot(
                            seeds[i],
                            vecs[j],
                        )

                if n == nmom:
                    break

                for j in range(self.nmo):

                    Hvec = self._apply_h(
                        vecs[j],
                        sector,
                    )

                    vecs[j] = (
                        E0 * vecs[j]
                        - Hvec
                    )

        # ==================================================
        # Alpha particle
        # ==================================================

        if self.nalpha < self.nmo:

            sector = (
                self.nalpha + 1,
                self.nbeta,
            )

            seeds = [
                addons.cre_a(
                    ci0,
                    self.nmo,
                    self.nelec,
                    j,
                )
                for j in range(self.nmo)
            ]

            vecs = [
                x.copy()
                for x in seeds
            ]

            for n in range(nmom + 1):

                for i in range(self.nmo):
                    for j in range(self.nmo):

                        particle_alpha[
                            n, i, j
                        ] = np.vdot(
                            seeds[i],
                            vecs[j],
                        )

                if n == nmom:
                    break

                for j in range(self.nmo):

                    Hvec = self._apply_h(
                        vecs[j],
                        sector,
                    )

                    vecs[j] = (
                        Hvec
                        - E0 * vecs[j]
                    )

        # ==================================================
        # Beta particle
        # ==================================================

        if self.nbeta < self.nmo:

            sector = (
                self.nalpha,
                self.nbeta + 1,
            )

            seeds = [
                addons.cre_b(
                    ci0,
                    self.nmo,
                    self.nelec,
                    j,
                )
                for j in range(self.nmo)
            ]

            vecs = [
                x.copy()
                for x in seeds
            ]

            for n in range(nmom + 1):

                for i in range(self.nmo):
                    for j in range(self.nmo):

                        particle_beta[
                            n, i, j
                        ] = np.vdot(
                            seeds[i],
                            vecs[j],
                        )

                if n == nmom:
                    break

                for j in range(self.nmo):

                    Hvec = self._apply_h(
                        vecs[j],
                        sector,
                    )

                    vecs[j] = (
                        Hvec
                        - E0 * vecs[j]
                    )

        hole = (
            hole_alpha
            + hole_beta
        )

        particle = (
            particle_alpha
            + particle_beta
        )

        return {
            "E0_total":
                self.e0_total,

            "E0_electronic":
                self.e0_electronic,

            "hole_moments_alpha":
                np.real_if_close(
                    hole_alpha
                ),

            "hole_moments_beta":
                np.real_if_close(
                    hole_beta
                ),

            "particle_moments_alpha":
                np.real_if_close(
                    particle_alpha
                ),

            "particle_moments_beta":
                np.real_if_close(
                    particle_beta
                ),

            "hole_moments":
                np.real_if_close(
                    hole
                ),

            "particle_moments":
                np.real_if_close(
                    particle
                ),
        }

In [6]:
mol = gto.Mole(
    atom = 'H 0 0 0; H 0 0 0.74', 
    basis = 'sto-3g', # minimal basis set
    spin = 0 # singlet state
)

# perform Hartree-Fock calculation
mf = scf.RHF(mol)
mf.kernel()

fci_engine = FCISpectralMoments(
    mf
)

fci_result = fci_engine.run_fci()

fci_moments = (
    fci_engine.compute_moments(
        nmom=10
    )
)

print(
    "FCI energy:",
    fci_result["fci_energy"]
)

print(
    "Hole moment 0:"
)

print(
    fci_moments[
        "hole_moments"
    ]
)

converged SCF energy = -1.11675930739643
FCI energy: -1.137283834488502
Hole moment 0:
[[[ 1.97466775e+00 -3.28040650e-16]
  [-3.28040650e-16  2.53322530e-02]]

 [[-1.18298077e+00  1.75299469e-16]
  [ 1.52926522e-16 -3.48906015e-02]]

 [[ 7.08698211e-01 -7.57878333e-17]
  [-3.15699921e-17  4.80554997e-02]]

 [[-4.24565781e-01  5.14341011e-18]
  [-6.37882140e-17 -6.61877685e-02]]

 [[ 2.54348183e-01  5.23688028e-17]
  [ 1.52119983e-16  9.11616926e-02]]

 [[-1.52374499e-01 -1.07745524e-16]
  [-2.48016512e-16 -1.25558761e-01]]

 [[ 9.12842692e-02  1.69737343e-16]
  [ 3.64661686e-16  1.72934508e-01]]

 [[-5.46864328e-02 -2.46565269e-16]
  [-5.16072582e-16 -2.38186039e-01]]

 [[ 3.27614599e-02  3.47256977e-16]
  [ 7.19074203e-16  3.28058235e-01]]

 [[-1.96266826e-02 -4.82871407e-16]
  [-9.95353644e-16 -4.51840947e-01]]

 [[ 1.17579213e-02  6.67816489e-16]
  [ 1.37389055e-15  6.22329268e-01]]]


Initialize <pyscf.gto.mole.Mole object at 0x117e61950> in <pyscf.scf.hf.RHF object at 0x117bd1f30>


In [ ]:
from pyscf import mcscf


class CASSpectralMomentsPySCF:
    """
    Exact CASCI spectral moments in PySCF.

    This is the exact benchmark corresponding to a chosen
    CAS(ne, no) Hamiltonian.

    The core orbitals remain frozen and the external orbitals
    remain unoccupied. FCI is performed within the active space.
    """

    def __init__(
        self,
        mf,
        ncas,
        nelecas,
        mo_coeff=None,
    ):

        self.mf = mf
        self.mol = mf.mol

        self.ncas = int(ncas)

        self.nelec = (
            int(nelecas[0]),
            int(nelecas[1]),
        )

        self.nalpha = self.nelec[0]
        self.nbeta = self.nelec[1]

        self.nmo = self.ncas

        if mo_coeff is None:
            mo_coeff = mf.mo_coeff

        self.mo_coeff = mo_coeff

        # --------------------------------------------------
        # CASCI object
        # --------------------------------------------------

        self.cas = mcscf.CASCI(
            mf,
            self.ncas,
            self.nelec,
        )

        # --------------------------------------------------
        # Effective active-space Hamiltonian
        # --------------------------------------------------

        self.h1, self.ecore = (
            self.cas.get_h1eff(
                self.mo_coeff
            )
        )

        eri = self.cas.get_h2eff(
            self.mo_coeff
        )

        self.eri = ao2mo.restore(
            1,
            eri,
            self.ncas,
        )

        self.e0_total = None
        self.e0_electronic = None
        self.ci0 = None

    # ======================================================
    # Exact CASCI
    # ======================================================

    def run_casci(self):

        result = self.cas.kernel(
            self.mo_coeff
        )

        e0 = result[0]
        ci0 = result[1]

        if not self.cas.converged:
            raise RuntimeError(
                "CASCI did not converge."
            )

        self.e0_total = float(e0)

        self.e0_electronic = (
            self.e0_total
            - self.ecore
        )

        self.ci0 = ci0

        return {
            "casci_energy":
                self.e0_total,

            "casci_active_energy":
                self.e0_electronic,

            "casci_ci":
                self.ci0,

            "ecore":
                self.ecore,
        }

    # ======================================================
    # Hamiltonian action
    # ======================================================

    def _apply_h(
        self,
        ci_vec,
        nelec_sector,
    ):

        h2eff = direct_spin1.absorb_h1e(
            self.h1,
            self.eri,
            self.nmo,
            nelec_sector,
            fac=0.5,
        )

        return direct_spin1.contract_2e(
            h2eff,
            ci_vec,
            self.nmo,
            nelec_sector,
        )

    # ======================================================
    # Spectral moments
    # ======================================================

    def compute_moments(
        self,
        nmom=10,
    ):

        if self.ci0 is None:
            self.run_casci()

        ci0 = self.ci0
        E0 = self.e0_electronic

        shape = (
            nmom + 1,
            self.nmo,
            self.nmo,
        )

        hole_alpha = np.zeros(
            shape,
            dtype=complex,
        )

        hole_beta = np.zeros(
            shape,
            dtype=complex,
        )

        particle_alpha = np.zeros(
            shape,
            dtype=complex,
        )

        particle_beta = np.zeros(
            shape,
            dtype=complex,
        )

        # ==================================================
        # Alpha hole
        # ==================================================

        if self.nalpha > 0:

            sector = (
                self.nalpha - 1,
                self.nbeta,
            )

            seeds = [
                addons.des_a(
                    ci0,
                    self.nmo,
                    self.nelec,
                    j,
                )
                for j in range(self.nmo)
            ]

            vecs = [
                x.copy()
                for x in seeds
            ]

            for n in range(nmom + 1):

                for i in range(self.nmo):
                    for j in range(self.nmo):

                        hole_alpha[
                            n, i, j
                        ] = np.vdot(
                            seeds[i],
                            vecs[j],
                        )

                if n == nmom:
                    break

                for j in range(self.nmo):

                    Hvec = self._apply_h(
                        vecs[j],
                        sector,
                    )

                    vecs[j] = (
                        E0 * vecs[j]
                        - Hvec
                    )

        # ==================================================
        # Beta hole
        # ==================================================

        if self.nbeta > 0:

            sector = (
                self.nalpha,
                self.nbeta - 1,
            )

            seeds = [
                addons.des_b(
                    ci0,
                    self.nmo,
                    self.nelec,
                    j,
                )
                for j in range(self.nmo)
            ]

            vecs = [
                x.copy()
                for x in seeds
            ]

            for n in range(nmom + 1):

                for i in range(self.nmo):
                    for j in range(self.nmo):

                        hole_beta[
                            n, i, j
                        ] = np.vdot(
                            seeds[i],
                            vecs[j],
                        )

                if n == nmom:
                    break

                for j in range(self.nmo):

                    Hvec = self._apply_h(
                        vecs[j],
                        sector,
                    )

                    vecs[j] = (
                        E0 * vecs[j]
                        - Hvec
                    )

        # ==================================================
        # Alpha particle
        # ==================================================

        if self.nalpha < self.nmo:

            sector = (
                self.nalpha + 1,
                self.nbeta,
            )

            seeds = [
                addons.cre_a(
                    ci0,
                    self.nmo,
                    self.nelec,
                    j,
                )
                for j in range(self.nmo)
            ]

            vecs = [
                x.copy()
                for x in seeds
            ]

            for n in range(nmom + 1):

                for i in range(self.nmo):
                    for j in range(self.nmo):

                        particle_alpha[
                            n, i, j
                        ] = np.vdot(
                            seeds[i],
                            vecs[j],
                        )

                if n == nmom:
                    break

                for j in range(self.nmo):

                    Hvec = self._apply_h(
                        vecs[j],
                        sector,
                    )

                    vecs[j] = (
                        Hvec
                        - E0 * vecs[j]
                    )

        # ==================================================
        # Beta particle
        # ==================================================

        if self.nbeta < self.nmo:

            sector = (
                self.nalpha,
                self.nbeta + 1,
            )

            seeds = [
                addons.cre_b(
                    ci0,
                    self.nmo,
                    self.nelec,
                    j,
                )
                for j in range(self.nmo)
            ]

            vecs = [
                x.copy()
                for x in seeds
            ]

            for n in range(nmom + 1):

                for i in range(self.nmo):
                    for j in range(self.nmo):

                        particle_beta[
                            n, i, j
                        ] = np.vdot(
                            seeds[i],
                            vecs[j],
                        )

                if n == nmom:
                    break

                for j in range(self.nmo):

                    Hvec = self._apply_h(
                        vecs[j],
                        sector,
                    )

                    vecs[j] = (
                        Hvec
                        - E0 * vecs[j]
                    )

        hole = (
            hole_alpha
            + hole_beta
        )

        particle = (
            particle_alpha
            + particle_beta
        )

        return {
            "E0_total":
                self.e0_total,

            "E0_active":
                self.e0_electronic,

            "ecore":
                self.ecore,

            "hole_moments_alpha":
                np.real_if_close(
                    hole_alpha
                ),

            "hole_moments_beta":
                np.real_if_close(
                    hole_beta
                ),

            "particle_moments_alpha":
                np.real_if_close(
                    particle_alpha
                ),

            "particle_moments_beta":
                np.real_if_close(
                    particle_beta
                ),

            "hole_moments":
                np.real_if_close(
                    hole
                ),

            "particle_moments":
                np.real_if_close(
                    particle
                ),
        }